# Potential-Mixing Monotonicity Experiments

Tests the hypothesis: mixing two potentials $V=V_1+g\cdot V_2$, how monotonic is the resulting $(n_s,r)$-vs-$g$ curve, and does that depend on how *different* $V_1$ and $V_2$ are (using each potential's own scale-type log-slope / local $\varepsilon_v$-order as the distance measure)?

Includes: the core framework, a quantitative non-monotonicity index (total-variation / net-variation), robust calibration (handles spurious boundary crossings and genuine graceful-exit failures), self-pairing tests, and the full pairwise table across 6 representative potentials.

## Core inflation framework

In [1]:
import numpy as np
from scipy.optimize import brentq
from scipy import integrate
from scipy.integrate import solve_ivp

"""
Reusable single-field inflation numerical framework.
"""


def background_eqn(N, fields, potential):
    phi, dphidN = fields
    v, dvdphi = potential(phi)
    d2phidN2 = -(3. - 0.5 * (dphidN * dphidN)) * dphidN \
               - (6. - (dphidN * dphidN)) * dvdphi / (2. * v)
    return [dphidN, d2phidN2]


def Hubble(phi_N, V):
    return np.sqrt(V / (3 - 0.5 * (phi_N * phi_N)))


def epsilon1(phi_N):
    return 0.5 * phi_N * phi_N


def stop_integration(N, y):
    return 0.5 * (y[1] ** 2) - 1
stop_integration.terminal = True
stop_integration.direction = 0


def find_nearest(array, value):
    array = np.asarray(array)
    idx = (np.abs(array - value)).argmin()
    return idx, array[idx]


def z_of(a, phi_N):
    return a * phi_N


def solve_background(potential, ph_initial, t_end=90.0, coarse_step=1e-2,
                      fine_step=5e-4, method='RK45'):
    efolds_fine = np.arange(0, t_end, fine_step)
    sol = solve_ivp(lambda N, y: background_eqn(N, y, potential),
                     [0, t_end], ph_initial, method=method,
                     dense_output=True, events=(stop_integration,),
                     vectorized=False, max_step=np.inf)

    if len(sol.t_events[0]) > 0:
        Nend = sol.t_events[0][0]
    else:
        Nend = sol.t[-1]

    mask = efolds_fine < Nend
    efolds = efolds_fine[mask]
    y = sol.sol(efolds)
    phi, dphi = y[0], y[1]

    pot, dpotdphi = potential(phi)
    Hub = Hubble(dphi, pot)

    return dict(efolds=efolds, phi=phi, dphi=dphi, pot=pot,
                dpotdphi=dpotdphi, Hub=Hub, Nend=Nend,
                inflation_completed=len(sol.t_events[0]) > 0)


def pert_eqn(y, N, k, ai, potential):
    phi, dphidN, Rk_re, Rk_re_N, Rk_im, Rk_im_N, hk_re, hk_re_N, hk_im, hk_im_N = y
    v, dvdphi = potential(phi)
    d2phidN2 = -(3. - 0.5 * (dphidN * dphidN)) * dphidN \
               - (6. - (dphidN * dphidN)) * dvdphi / (2. * v)
    a = ai * np.exp(N)
    H = np.sqrt(v / (3 - 0.5 * (dphidN * dphidN)))
    zN = a * (dphidN + d2phidN2)
    zval = z_of(a, dphidN)

    Rk_re_NN = -(1 - (dphidN ** 2) * 0.5 + 2.0 * (zN / zval)) * Rk_re_N - ((k / (a * H)) ** 2) * Rk_re
    Rk_im_NN = -(1 - (dphidN ** 2) * 0.5 + 2.0 * (zN / zval)) * Rk_im_N - ((k / (a * H)) ** 2) * Rk_im
    hk_re_NN = -(3 - (dphidN ** 2) * 0.5) * hk_re_N - ((k / (a * H)) ** 2) * hk_re
    hk_im_NN = -(3 - (dphidN ** 2) * 0.5) * hk_im_N - ((k / (a * H)) ** 2) * hk_im

    return [dphidN, d2phidN2, Rk_re_N, Rk_re_NN, Rk_im_N, Rk_im_NN,
            hk_re_N, hk_re_NN, hk_im_N, hk_im_NN]


def boundary(k, t, aH, Nic_cond, Nshs_cond):
    ic_diff = k - Nic_cond * aH
    shs_diff = k - Nshs_cond * aH
    idx_ic, _ = find_nearest(ic_diff, 0)
    idx_shs, _ = find_nearest(shs_diff, 0)
    return idx_ic, t[idx_ic], t[idx_shs]


def initial_condition(idx_ic, k, efolds, phi, dphi, ai, Hub, potential):
    phi_ic = phi[idx_ic]
    phiN_ic = dphi[idx_ic]
    a_ic = ai * np.exp(efolds[idx_ic])
    z_ic = z_of(a_ic, phiN_ic)
    H_ic = Hub[idx_ic]
    _, phiNN_ic = background_eqn(efolds[idx_ic], [phi_ic, phiN_ic], potential)

    Rk_re_ic = (1 / (np.sqrt(2 * k))) / z_ic
    Rk_im_ic = 0
    Rk_re_N_ic = -Rk_re_ic * ((phiNN_ic / phiN_ic) + 1)
    Rk_im_N_ic = -np.sqrt(k / 2) / (a_ic * H_ic * z_ic)
    hk_re_ic = (1 / (np.sqrt(2 * k))) / a_ic
    hk_im_ic = 0
    hk_re_N_ic = -hk_re_ic
    hk_im_N_ic = -np.sqrt(k / 2) / (a_ic * H_ic * a_ic)

    return [phi_ic, phiN_ic, Rk_re_ic, Rk_re_N_ic, Rk_im_ic, Rk_im_N_ic,
            hk_re_ic, hk_re_N_ic, hk_im_ic, hk_im_N_ic]


def Psk(kk, Rk):
    Rk_re, Rk_im = Rk
    return kk ** 3 / (2e0 * np.pi ** 2) * ((Rk_re * Rk_re) + (Rk_im * Rk_im))


def Pth(kk, hk):
    hk_re, hk_im = hk
    return 8.0 * kk ** 3 / (2e0 * np.pi ** 2) * ((hk_re * hk_re) + (hk_im * hk_im))


class InflationRun:
    def __init__(self, potential, bg, kpivot=0.05, Nic_cond=100, Nshs_cond=1e-5):
        self.potential = potential
        self.bg = bg
        self.efolds = bg['efolds']
        self.phi = bg['phi']
        self.dphi = bg['dphi']
        self.Hub = bg['Hub']
        self.Nend = bg['Nend']
        self.Nic_cond = Nic_cond
        self.Nshs_cond = Nshs_cond

        idx50, Ne50 = find_nearest(self.efolds, self.Nend - 50)
        self.idx50, self.Ne50 = idx50, Ne50
        self.ai = kpivot / (np.exp(Ne50) * self.Hub[idx50])
        self.kpivot = kpivot

    def compute_pps(self, kk, atol=1e-11):
        idx_ic, Nic, Nshs = boundary(kk, self.efolds,
                                      self.ai * np.exp(self.efolds) * self.Hub,
                                      self.Nic_cond, self.Nshs_cond)
        efolds_pert = np.arange(Nic, Nshs, 0.01)
        pert_ic = initial_condition(idx_ic, kk, self.efolds, self.phi, self.dphi,
                                     self.ai, self.Hub, self.potential)
        sol_pert = integrate.odeint(pert_eqn, pert_ic, efolds_pert,
                                     args=(kk, self.ai, self.potential), atol=atol)
        Rkshs = [sol_pert[-1, 2], sol_pert[-1, 4]]
        hkshs = [sol_pert[-1, 6], sol_pert[-1, 8]]
        return Psk(kk, Rkshs), Pth(kk, hkshs), efolds_pert, sol_pert, Nic, Nshs

    def spectrum(self, karray, atol=1e-11):
        PS = np.zeros(len(karray))
        PT = np.zeros(len(karray))
        for i, kk in enumerate(karray):
            PS[i], PT[i], _, _, _, _ = self.compute_pps(kk, atol=atol)
        return PS, PT


## Mixing-experiment machinery

In [2]:
TARGET_NEND = 65.0
V_FLOOR_RATIO = 1e-8  # bail out if V drops below this fraction of its initial value


def _stop_eps1(N, y):
    return 0.5*(y[1]**2) - 1
_stop_eps1.terminal = True
_stop_eps1.direction = 0


def solve_background_safe(potential, ph_initial, t_end=150.0, dense=True, fine_step=5e-4):
    """
    Like solve_background above, but with two robustness fixes needed for
    arbitrary MIXED potentials that individually-validated single-potential
    runs never needed:

    1. A second stopping event that fires if V collapses toward zero (below
       V_FLOOR_RATIO times its initial value). Some mixed potentials (e.g.
       quartic + natural, which BOTH vanish at phi=0) have a genuine V->0
       singularity the field can roll toward. The Klein-Gordon equation has
       a 1/V term, so as the trajectory approaches that point the adaptive
       integrator can be driven to pathologically tiny step sizes trying to
       resolve it -- this manifested as a runtime/RAM blowup, not a clean
       "ran to t_end, never completed" like the quad+hilltop graceful-exit
       case (which had a safe nonzero potential floor). This event stops
       the integration cleanly the moment that starts happening, rather
       than letting the solver grind indefinitely.
    2. dense_output is OFF by default for lightweight calls -- used during
       bracket-SCANNING and root-finding, where we only need Nend, not the
       full trajectory. Dense output's per-step interpolant storage is the
       main memory/time cost when many steps are needed, and scanning calls
       this many times per bracket search; the final calibrated solve
       (after brentq converges) always uses dense=True.
    """
    V0, _ = potential(ph_initial[0])
    v_floor = V_FLOOR_RATIO * abs(V0)

    def stop_v_floor(N, y):
        v, _ = potential(y[0])
        return v - v_floor
    stop_v_floor.terminal = True
    stop_v_floor.direction = 0

    sol = solve_ivp(lambda N, y: background_eqn(N, y, potential),
                     [0, t_end], ph_initial, method='RK45',
                     dense_output=dense, events=(_stop_eps1, stop_v_floor),
                     max_step=np.inf)

    eps1_fired = len(sol.t_events[0]) > 0
    vfloor_fired = len(sol.t_events[1]) > 0

    if eps1_fired:
        Nend = sol.t_events[0][0]
        completed = True
    elif vfloor_fired:
        Nend = sol.t_events[1][0]
        completed = False
    else:
        Nend = sol.t[-1]
        completed = False

    if not dense:
        return dict(Nend=Nend, inflation_completed=completed)

    efolds_fine = np.arange(0, Nend, fine_step)
    mask = efolds_fine < Nend
    efolds = efolds_fine[mask]
    y = sol.sol(efolds)
    phi, dphi = y[0], y[1]
    pot, dpotdphi = potential(phi)
    Hub = Hubble(dphi, pot)
    return dict(efolds=efolds, phi=phi, dphi=dphi, pot=pot, dpotdphi=dpotdphi,
                Hub=Hub, Nend=Nend, inflation_completed=completed)


def make_mixed_potential(V1_fn, V2_fn, g):
    """V1_fn, V2_fn: functions phi -> (V, dV/dphi). Returns V = V1 + g*V2, with dV via sum rule."""
    def pot(phi):
        v1, dv1 = V1_fn(phi)
        v2, dv2 = V2_fn(phi)
        return v1 + g*v2, dv1 + g*dv2
    return pot


def eps_eta_numeric(pot, phi, h=1e-6):
    """Finite-difference eps_v, eta_v -- avoids hand-derivation errors for arbitrary mixes."""
    V0, dV0 = pot(phi)
    Vp, _ = pot(phi+h)
    Vm, _ = pot(phi-h)
    d2V = (Vp - 2*V0 + Vm)/h**2
    eps_v = 0.5*(dV0/V0)**2
    eta_v = d2V/V0
    return eps_v, eta_v


def Nend_of_phi(phi_i, pot, dense=False):
    ph = [phi_i, None]
    V, dV = pot(phi_i)
    ph[1] = -dV/V
    return solve_background_safe(pot, ph, t_end=150, dense=dense)['Nend']


def find_all_brackets(pot, phi_lo, phi_hi, target_nend=TARGET_NEND, n_scan=40):
    """Returns ALL sign-change brackets found in range, in order, so the
    caller can skip spurious ones (e.g. near a bounded potential's domain
    edge) and try the next genuine crossing. Uses dense=False (cheap,
    Nend-only solves) since only the scalar Nend is needed here."""
    grid = np.linspace(phi_lo, phi_hi, n_scan)
    vals = []
    for p in grid:
        try:
            vals.append(Nend_of_phi(p, pot, dense=False))
        except Exception:
            vals.append(np.nan)
    vals = np.array(vals)
    diffs = vals - target_nend
    brackets = []
    for i in range(len(grid)-1):
        if np.isnan(diffs[i]) or np.isnan(diffs[i+1]):
            continue
        if diffs[i] == 0:
            brackets.append((grid[i], grid[i+1]))
        elif diffs[i]*diffs[i+1] < 0:
            brackets.append((grid[i], grid[i+1]))
    if not brackets:
        raise RuntimeError(f"No bracket found for target_nend={target_nend} in [{phi_lo},{phi_hi}]")
    return brackets


def calibrate_ns_r(pot, phi_i_guess_lo, phi_i_guess_hi, target_nend=TARGET_NEND, n_scan=20):
    """Root-find phi_i for exact N_end, then return (phi_i, ns, r) at the pivot.
    Tries every crossing found (in order) until one gives a physically
    sensible, genuinely-completed result, so a spurious crossing near a
    bounded potential's domain edge -- or a V->0 singularity that merely
    happens to cross target_nend on the way down -- doesn't silently poison
    the answer. Root-finding itself uses dense=False (fast); only the final
    accepted solve is redone with dense=True for the full trajectory."""
    brackets = find_all_brackets(pot, phi_i_guess_lo, phi_i_guess_hi, target_nend, n_scan=n_scan)
    last_err = None
    for lo, hi in brackets:
        try:
            phi_i = brentq(lambda p: Nend_of_phi(p, pot, dense=False)-target_nend, lo, hi, xtol=1e-6)
            ph = [phi_i, None]
            V, dV = pot(phi_i)
            ph[1] = -dV/V
            bg = solve_background_safe(pot, ph, t_end=150, dense=True, fine_step=5e-4)
            if not bg['inflation_completed'] or abs(bg['Nend'] - target_nend) > 0.1:
                raise RuntimeError(f"Nend mismatch/incomplete: got {bg['Nend']}, completed={bg['inflation_completed']}")
            run = InflationRun(pot, bg, kpivot=0.05, Nic_cond=100)
            phi_star = bg['phi'][run.idx50]
            eps_v, eta_v = eps_eta_numeric(pot, phi_star)
            ns = 1 - 6*eps_v + 2*eta_v
            r = 16*eps_v
            if not (0.3 < ns < 1.05) or not (0 <= r < 10) or eps_v < 0:
                raise RuntimeError(f"Unphysical result: ns={ns}, r={r}")
            return phi_i, ns, r
        except Exception as e:
            last_err = e
            continue
    raise RuntimeError(f"All {len(brackets)} bracket(s) failed validation. Last error: {last_err}")


def nonmonotonicity_index(y):
    """
    Total-variation / net-variation - 1.
    = 0 for a perfectly monotonic sequence (TV == |net change|).
    > 0 growing with how much the sequence reverses direction, scaled by
    how large those reversals are relative to the overall trend.
    Robust to sequences with near-zero net change (returns inf/0 sensibly).
    If ANY entry is NaN (meaning that g-value hit a graceful-exit failure --
    see run_mixing_sweep), returns +inf: a coupling range where inflation
    doesn't even complete is treated as maximally pathological, not just
    "non-monotonic" -- this is a deliberate modeling choice (Option 2),
    keeping every pair comparable on one scale rather than silently
    dropping failed points or reporting them separately.
    """
    y = np.asarray(y, dtype=float)
    if np.any(np.isnan(y)):
        return np.inf
    total_variation = np.sum(np.abs(np.diff(y)))
    net_variation = np.abs(y[-1] - y[0])
    if net_variation < 1e-30:
        return np.inf if total_variation > 1e-30 else 0.0
    return total_variation/net_variation - 1.0


def run_mixing_sweep(V1_fn, V2_fn, g_values, phi_i_bracket_fn, verbose=True):
    """
    Sweep g, calibrate (ns, r) at each point.
    phi_i_bracket_fn(g) -> (lo, hi) bracket for the root-finder, since the
    right bracket generally shifts as g changes.
    Any g that fails (no valid bracket, unphysical result, a genuine
    graceful-exit failure, or a V->0 singularity -- inflation never
    completes for ANY phi_i tried) is recorded as NaN rather than crashing
    the whole sweep; NaN entries propagate to nonmonotonicity_index() as
    +inf (Option 2: a coupling that breaks graceful exit is treated as
    maximally pathological).
    Returns dict with g, ns, r arrays (NaN at failed points) and both
    non-monotonicity indices.
    """
    ns_vals, r_vals = [], []
    for g in g_values:
        pot = make_mixed_potential(V1_fn, V2_fn, g)
        lo, hi = phi_i_bracket_fn(g)
        try:
            phi_i, ns, r = calibrate_ns_r(pot, lo, hi)
            ns_vals.append(ns)
            r_vals.append(r)
            if verbose:
                print(f'  g={g:.4g}: phi_i={phi_i:.5f}  ns={ns:.4f}  r={r:.6f}')
        except Exception as e:
            ns_vals.append(np.nan)
            r_vals.append(np.nan)
            if verbose:
                print(f'  g={g:.4g}: FAILED (graceful-exit/singularity/calibration) -- {str(e)[:80]}')
    ns_vals = np.array(ns_vals)
    r_vals = np.array(r_vals)
    return dict(g=np.array(g_values), ns=ns_vals, r=r_vals,
                nonmono_ns=nonmonotonicity_index(ns_vals),
                nonmono_r=nonmonotonicity_index(r_vals))


def exp_g_values(g_min=0.01, g_max=100.0, n=5, include_zero=True):
    """Exponentially (log-)spaced g values, e.g. [0, 0.01, 0.1, 1, 10, 100]
    for defaults -- more sensible than linear spacing since interesting
    structure (peaks, singularities) can occur across many orders of
    magnitude in the coupling, not spread evenly on a linear scale.
    Available for any future sweep; the sweeps below keep their original,
    already-tuned g-value lists so results stay directly comparable."""
    vals = list(np.logspace(np.log10(g_min), np.log10(g_max), n))
    if include_zero:
        vals = [0.0] + vals
    return vals


## Sanity check: reproduce the three known cases (Pair A / quad+quartic / Pair B)
Confirms the nonmonotonicity_index gives 0 for the known-monotonic case and increasingly large values for the known non-monotonic cases.

In [3]:
pairA_r = np.array([0.001486,0.000495,0.000262,0.000133,0.000064,0.000046,0.000037,0.000033,0.000031])
pairA_ns = np.array([0.8882,0.8517,0.8329,0.8149,0.7966,0.7871,0.7824,0.7790,0.7785])
qm_r = np.array([0.164059,0.184581,0.252677,0.296284,0.335477,0.338611,0.336187,0.329031,0.326070,0.323210,0.321631])
qm_ns = np.array([0.9596,0.9591,0.9533,0.9467,0.9387,0.9376,0.9375,0.9385,0.9390,0.9394,0.9397])
pairB_r = np.array([0.161676,0.161680,0.161714,0.162046,0.163420,0.164942,0.167555,0.173191,0.176594,0.162773,0.073726])
pairB_ns = np.array([0.9595,0.9595,0.9595,0.9594,0.9597,0.9592,0.9585,0.9576,0.9596,0.9672,0.9970])

for name, r, ns in [('Pair A (close)', pairA_r, pairA_ns),
                     ('quad+quartic (medium)', qm_r, qm_ns),
                     ('Pair B (far)', pairB_r, pairB_ns)]:
    print(f'{name}: nonmono(r)={nonmonotonicity_index(r):.4f}  nonmono(ns)={nonmonotonicity_index(ns):.4f}')

Pair A (close): nonmono(r)=0.0000  nonmono(ns)=0.0000
quad+quartic (medium): nonmono(r)=0.2155  nonmono(ns)=0.2211
Pair B (far): nonmono(r)=0.3392  nonmono(ns)=0.1173


## Self-pairing test 1 (SCALE-type): hilltop($\mu=6$) + $g\cdot$hilltop($\mu=20$)
Prediction: same local power-order (both $x^2$-type) -> should stay MONOTONIC despite the huge width mismatch.

In [4]:
def hilltop_mu6(phi, mu=6.0):
    v = 1-(phi/mu)**2; dv = -2*phi/mu**2
    return v, dv
def hilltop_mu20(phi, mu=20.0):
    v = 1-(phi/mu)**2; dv = -2*phi/mu**2
    return v, dv

def bracket_selfpair1(g): return (1e-4, 19.999)
res_self1 = run_mixing_sweep(hilltop_mu6, hilltop_mu20,
                              [0.0, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0], bracket_selfpair1)
print(f"nonmono(ns)={res_self1['nonmono_ns']:.4f}   nonmono(r)={res_self1['nonmono_r']:.4f}")

  g=0: phi_i=0.10795  ns=0.8885  r=0.001486
  g=0.5: phi_i=0.37234  ns=0.9201  r=0.005350
  g=1: phi_i=0.71839  ns=0.9345  r=0.009787
  g=2: phi_i=1.45389  ns=0.9476  r=0.017357
  g=5: phi_i=3.24857  ns=0.9582  r=0.029816
  g=10: phi_i=5.03323  ns=0.9625  r=0.037853
  g=20: phi_i=6.74583  ns=0.9646  r=0.043400
  g=50: phi_i=8.38732  ns=0.9666  r=0.047482
nonmono(ns)=0.0000   nonmono(r)=-0.0000


## Self-pairing test 2 (SHAPE-type): monomial($p=2$) + $g\cdot$monomial($p=6$)
Prediction: different local power-order ($x^2$ vs $x^6$) -> should show NON-MONOTONIC behavior.

In [5]:
def monomial_p2(phi): return 0.5*phi**2, phi
def monomial_p6(phi): return phi**6/6.0, phi**5

def bracket_selfpair2(g): return (1e-4, 60.0)
res_self2 = run_mixing_sweep(monomial_p2, monomial_p6,
                              [0.0, 0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 20.0], bracket_selfpair2)
print(f"nonmono(ns)={res_self2['nonmono_ns']:.4f}   nonmono(r)={res_self2['nonmono_r']:.4f}")

/tmp/ipykernel_1778/48772506.py:14: RuntimeWarning: overflow encountered in scalar multiply
  d2phidN2 = -(3. - 0.5 * (dphidN * dphidN)) * dphidN \


  g=0: phi_i=16.05476  ns=0.9598  r=0.161676


/tmp/ipykernel_1778/48772506.py:15: RuntimeWarning: overflow encountered in scalar multiply
  - (6. - (dphidN * dphidN)) * dvdphi / (2. * v)


  g=0.001: phi_i=26.06174  ns=0.9062  r=0.566006
  g=0.01: phi_i=27.28251  ns=0.9155  r=0.509134
  g=0.05: phi_i=27.69638  ns=0.9184  r=0.489994
  g=0.1: phi_i=27.95296  ns=0.9198  r=0.478444
  g=0.5: phi_i=28.01702  ns=0.9203  r=0.475659
  g=1: phi_i=28.02379  ns=0.9210  r=0.475367
  g=5: phi_i=28.02907  ns=0.9214  r=0.475140
  g=20: phi_i=28.03005  ns=0.9204  r=0.475098
nonmono(ns)=0.7713   nonmono(r)=0.5801


## Full pairwise table: 6 representative potentials, 15 pairs

Uses Option 2 for graceful-exit failures: any $g$ where inflation never completes (for ANY $\phi_i$ tried) is recorded as NaN, which propagates to `nonmonotonicity_index` = $+\infty$ -- a coupling range that breaks graceful exit is treated as maximally pathological, keeping every pair comparable on one scale rather than needing a separate 'valid range' per pair.

In [6]:
def quadratic(phi): return 0.5*phi**2, phi
def quartic(phi): return 0.25*phi**4, phi**3
def hilltop(phi, mu=6.0):
    v = 1-(phi/mu)**2; dv = -2*phi/mu**2
    return v, dv
def plateau(phi, q=np.sqrt(2/3)):
    u = np.exp(-q*phi); v=(1-u)**2; dv=2*q*u*(1-u)
    return v, dv
def natural(phi, f=5.0):
    v = 1-np.cos(phi/f); dv = np.sin(phi/f)/f
    return v, dv
def symbreak(phi, mu=6.0):
    x = phi/mu; v=(1-x**2)**2; dv=2*(1-x**2)*(-2*x/mu)
    return v, dv

POTENTIALS = {
    'quadratic': quadratic, 'quartic': quartic, 'hilltop': hilltop,
    'plateau': plateau, 'natural': natural, 'symbreak': symbreak,
}

In [7]:
import itertools
import pickle

g_values = [0.0, 0.05, 0.2, 1.0, 5.0, 25.0]
names = list(POTENTIALS.keys())
pairwise_results = {}

for n1, n2 in itertools.combinations(names, 2):
    print(f'=== {n1} + g*{n2} ===')
    def bracket(g): return (1e-4, 100.0)
    res = run_mixing_sweep(POTENTIALS[n1], POTENTIALS[n2], g_values, bracket, verbose=False)
    pairwise_results[(n1, n2)] = (res['nonmono_ns'], res['nonmono_r'])
    print(f"  nonmono(ns)={res['nonmono_ns']:.4f}   nonmono(r)={res['nonmono_r']:.4f}")

with open('pairwise_results_final.pkl', 'wb') as f:
    pickle.dump(pairwise_results, f)

=== quadratic + g*quartic ===


/tmp/ipykernel_1778/48772506.py:14: RuntimeWarning: overflow encountered in scalar multiply
  d2phidN2 = -(3. - 0.5 * (dphidN * dphidN)) * dphidN \
/tmp/ipykernel_1778/48772506.py:15: RuntimeWarning: overflow encountered in scalar multiply
  - (6. - (dphidN * dphidN)) * dvdphi / (2. * v)


  nonmono(ns)=0.2613   nonmono(r)=0.2331
=== quadratic + g*hilltop ===
  nonmono(ns)=inf   nonmono(r)=inf
=== quadratic + g*plateau ===


/tmp/ipykernel_1778/2915481528.py:7: RuntimeWarning: overflow encountered in exp
  u = np.exp(-q*phi); v=(1-u)**2; dv=2*q*u*(1-u)
/tmp/ipykernel_1778/294558482.py:81: RuntimeWarning: invalid value encountered in scalar multiply
  return v1 + g*v2, dv1 + g*dv2
/tmp/ipykernel_1778/48772506.py:15: RuntimeWarning: invalid value encountered in scalar divide
  - (6. - (dphidN * dphidN)) * dvdphi / (2. * v)


  nonmono(ns)=0.2557   nonmono(r)=1.0382
=== quadratic + g*natural ===
  nonmono(ns)=1.6277   nonmono(r)=0.0000
=== quadratic + g*symbreak ===
  nonmono(ns)=inf   nonmono(r)=inf
=== quartic + g*hilltop ===
  nonmono(ns)=inf   nonmono(r)=inf
=== quartic + g*plateau ===
  nonmono(ns)=0.3857   nonmono(r)=0.0000
=== quartic + g*natural ===


/tmp/ipykernel_1778/2915481528.py:2: RuntimeWarning: overflow encountered in scalar power
  def quartic(phi): return 0.25*phi**4, phi**3
/tmp/ipykernel_1778/2915481528.py:10: RuntimeWarning: invalid value encountered in cos
  v = 1-np.cos(phi/f); dv = np.sin(phi/f)/f
/tmp/ipykernel_1778/2915481528.py:10: RuntimeWarning: invalid value encountered in sin
  v = 1-np.cos(phi/f); dv = np.sin(phi/f)/f


  nonmono(ns)=1.7285   nonmono(r)=0.0000
=== quartic + g*symbreak ===
  nonmono(ns)=inf   nonmono(r)=inf
=== hilltop + g*plateau ===
  nonmono(ns)=2.8286   nonmono(r)=0.0437
=== hilltop + g*natural ===
  nonmono(ns)=1.2184   nonmono(r)=0.8698
=== hilltop + g*symbreak ===
  nonmono(ns)=0.0000   nonmono(r)=0.0000
=== plateau + g*natural ===
  nonmono(ns)=3.4170   nonmono(r)=0.5232
=== plateau + g*symbreak ===
  nonmono(ns)=inf   nonmono(r)=inf
=== natural + g*symbreak ===


/tmp/ipykernel_1778/48772506.py:15: RuntimeWarning: divide by zero encountered in scalar divide
  - (6. - (dphidN * dphidN)) * dvdphi / (2. * v)
/tmp/ipykernel_1778/48772506.py:14: RuntimeWarning: invalid value encountered in scalar subtract
  d2phidN2 = -(3. - 0.5 * (dphidN * dphidN)) * dphidN \


  nonmono(ns)=inf   nonmono(r)=inf


In [8]:
print(f"{'pair':<28} {'nonmono(ns)':>12} {'nonmono(r)':>12}")
for (n1, n2), (nm_ns, nm_r) in sorted(pairwise_results.items(), key=lambda kv: kv[1][1]):
    print(f"{n1+' + '+n2:<28} {nm_ns:>12.4f} {nm_r:>12.4f}")

pair                          nonmono(ns)   nonmono(r)
quadratic + natural                1.6277       0.0000
quartic + plateau                  0.3857       0.0000
quartic + natural                  1.7285       0.0000
hilltop + symbreak                 0.0000       0.0000
hilltop + plateau                  2.8286       0.0437
quadratic + quartic                0.2613       0.2331
plateau + natural                  3.4170       0.5232
hilltop + natural                  1.2184       0.8698
quadratic + plateau                0.2557       1.0382
quadratic + hilltop                   inf          inf
quadratic + symbreak                  inf          inf
quartic + hilltop                     inf          inf
quartic + symbreak                    inf          inf
plateau + symbreak                    inf          inf
natural + symbreak                    inf          inf
